# Databricks Release Notes ETL

Ingests release notes for **AWS**, **Azure**, and **GCP** from the official Databricks RSS feeds.
Upcoming features are identified via the `whatscoming` RSS category.

**Pipeline:** RSS feeds → Python parsing → Spark DataFrame → MERGE into Delta table (content-hash change detection)

**Outputs per run:** counts of inserted, updated, and untouched rows.

## 1. Configuration

Imports, widget parameters, feed URLs, and HTTP session setup.

In [ ]:
import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry
import xml.etree.ElementTree as ET
import re
import hashlib
from datetime import datetime, date
from html.parser import HTMLParser

from pyspark.sql import Row
from pyspark.sql.types import StructType, StructField, StringType, DateType
from pyspark.sql.functions import (
    col, current_timestamp, lit, md5, concat_ws, coalesce,
)

# Widget parameters — passed by the DAB job or set interactively
dbutils.widgets.text("catalog", "main", "Catalog Name")
dbutils.widgets.text("schema", "databricks_release_notes", "Schema Name")

CATALOG = dbutils.widgets.get("catalog")
SCHEMA = dbutils.widgets.get("schema")
TARGET_TABLE = f"{CATALOG}.{SCHEMA}.release_notes"

# One RSS feed per cloud provider
RSS_FEEDS = {
    "AWS":   "https://docs.databricks.com/aws/en/feed.xml",
    "Azure": "https://learn.microsoft.com/en-us/azure/databricks/feed.xml",
    "GCP":   "https://docs.databricks.com/gcp/en/feed.xml",
}

# Columns whose values form the content hash for change detection
CONTENT_COLS = [
    "cloud", "title", "description", "url",
    "status", "feature", "impact", "categories", "source",
]

# HTTP session with retry on transient errors (429 / 5xx)
_session = requests.Session()
_session.mount("https://", HTTPAdapter(max_retries=Retry(
    total=3, backoff_factor=1.0, status_forcelist=[429, 500, 502, 503, 504],
)))

print(f"✅ Target: {TARGET_TABLE}")
for cloud, url in RSS_FEEDS.items():
    print(f"   {cloud}: {url}")

## 2. Helper Functions

- **Text cleaning:** strips HTML tags and collapses whitespace.
- **Classification:** derives `status` (GA, Public Preview, Upcoming, …) and `impact` (High/Medium/Low) from the title and RSS categories.
- **Date parsing:** handles multiple date formats found across the three feeds.

In [ ]:
class _HTMLStripper(HTMLParser):
    """Lightweight HTML-to-text converter using stdlib html.parser."""

    def __init__(self):
        super().__init__()
        self._parts: list[str] = []

    def handle_data(self, data):
        self._parts.append(data)

    def get_text(self) -> str:
        return " ".join(self._parts)


def _strip_html(html: str) -> str:
    """Remove HTML tags, keeping only visible text."""
    s = _HTMLStripper()
    s.feed(html)
    return s.get_text()


def _clean(text: str, max_len: int | None = None) -> str:
    """Strip HTML, collapse whitespace, optionally truncate."""
    out = re.sub(r"\s+", " ", _strip_html(text)).strip()
    return out[:max_len].rstrip() if max_len and len(out) > max_len else out


# Regex patterns matched against the title to determine release status
_STATUS_PATTERNS = [
    (r"generally available|\(ga\)", "GA"),
    (r"public preview",             "Public Preview"),
    (r"private preview",            "Private Preview"),
    (r"\bbeta\b",                   "Beta"),
    (r"deprecated|deprecation|retirement", "Deprecated"),
    (r"breaking change|behavioral change|behavior change", "Breaking Change"),
    (r"maintenance update",         "Maintenance"),
]

# RSS categories that describe metadata, not a product feature
_META_CATEGORIES = frozenset({
    "product", "whatscoming", "aibi", "dbsql", "genie",
    "geniespaces", "databricksone", "release notes",
    "release-notes", "what's coming",
})


def _classify(title: str, categories: list[str]) -> tuple[str, str]:
    """Derive (status, impact) from the title text and RSS categories.

    Status priority: whatscoming category → regex match on title → default 'Released'.
    Impact: High for breaking/deprecated, Medium for GA/Upcoming, Low otherwise.
    """
    lower_cats = {c.lower() for c in categories}
    if "whatscoming" in lower_cats or "what's coming" in lower_cats:
        status = "Upcoming"
    else:
        status = "Released"
        lower = title.lower()
        for pattern, s in _STATUS_PATTERNS:
            if re.search(pattern, lower):
                status = s
                break

    if status in ("Breaking Change", "Deprecated"):
        impact = "High"
    elif status == "Upcoming" and re.search(r"breaking|behavior", title, re.I):
        impact = "High"
    elif status in ("GA", "Upcoming", "Released"):
        impact = "Medium"
    else:
        impact = "Low"

    return status, impact


# Date formats encountered across the three RSS feeds
_DATE_FMTS = (
    "%a, %d %b %Y %H:%M:%S %Z",
    "%a, %d %b %Y %H:%M:%S %z",
    "%Y-%m-%dT%H:%M:%S%z",
    "%Y-%m-%d",
)


def _parse_date(raw: str) -> date:
    """Best-effort date parsing; falls back to today if all formats fail."""
    s = raw.strip()
    for fmt in _DATE_FMTS:
        try:
            return datetime.strptime(s, fmt).date()
        except ValueError:
            continue
    # Some feeds append a non-standard timezone abbreviation
    try:
        return datetime.strptime(
            re.sub(r"\s+[A-Z]{2,4}$", "", s), "%a, %d %b %Y %H:%M:%S"
        ).date()
    except ValueError:
        return date.today()


print("✅ Helpers defined")

## 3. RSS Feed Parser

Fetches a single RSS feed, extracts `<item>` elements, and returns a list of dicts ready for Spark ingestion.
Each entry gets a deterministic `id` (MD5 of `cloud:guid`) for idempotent upserts.

In [ ]:
def parse_rss(url: str, cloud: str) -> list[dict]:
    """Fetch one RSS feed and return parsed release-note dicts."""
    print(f"  {cloud}: {url}")
    resp = _session.get(url, timeout=30)
    resp.raise_for_status()
    root = ET.fromstring(resp.content)

    entries = []
    for item in root.findall(".//item"):
        title = (item.findtext("title") or "").strip()
        if not title:
            continue

        link = (item.findtext("link") or "").strip()
        guid = (item.findtext("guid") or link).strip()
        categories = [c.text.strip() for c in item.findall("category") if c.text]

        status, impact = _classify(title, categories)
        # Use the first non-meta category as the feature area
        candidates = [c for c in categories if c.lower() not in _META_CATEGORIES]
        feature = candidates[0] if candidates else "General"

        entries.append(dict(
            id          = hashlib.md5(f"{cloud}:{guid}".encode()).hexdigest(),
            cloud       = cloud,
            title       = title,
            description = _clean(item.findtext("description") or "", max_len=2000),
            pub_date    = _parse_date(item.findtext("pubDate") or ""),
            url         = link,
            status      = status,
            feature     = feature,
            impact      = impact,
            categories  = ", ".join(categories),
            source      = "rss_feed",
        ))

    print(f"    → {len(entries)} entries")
    return entries


print("✅ RSS parser defined")

## 4. Fetch & Stage

Iterates over all configured RSS feeds, builds a Spark DataFrame with an explicit schema, and registers it as a temp view for the subsequent MERGE.

In [ ]:
print("Fetching RSS feeds …")
all_entries: list[dict] = []
for cloud, url in RSS_FEEDS.items():
    all_entries.extend(parse_rss(url, cloud))

print(f"\nTotal entries: {len(all_entries)}")

# Explicit schema ensures correct types regardless of feed content
df_schema = StructType([
    StructField("id",          StringType(), False),
    StructField("cloud",       StringType(), False),
    StructField("title",       StringType(), False),
    StructField("description", StringType(), True),
    StructField("pub_date",    DateType(),   True),
    StructField("url",         StringType(), True),
    StructField("status",      StringType(), True),
    StructField("feature",     StringType(), True),
    StructField("impact",      StringType(), True),
    StructField("categories",  StringType(), True),
    StructField("source",      StringType(), True),
])

staged_df = (
    spark.createDataFrame([Row(**e) for e in all_entries], schema=df_schema)
    .withColumn("ingested_at", current_timestamp())
    .filter(col("title").isNotNull() & (col("title") != ""))
    .withColumn("content_hash", md5(concat_ws("|", *[coalesce(col(c), lit("")) for c in CONTENT_COLS])))
)

staged_df.createOrReplaceTempView("release_notes_staged")
print(f"✅ Staged {staged_df.count()} entries")

## 5. MERGE into Delta Table

Creates the target schema/table if they don't exist, then performs a MERGE:
- **INSERT** new rows (id not in target).
- **UPDATE** existing rows only when the content hash differs (avoids unnecessary writes).
- Rows with unchanged content are left untouched.

In [ ]:
# Auto-provision schema and table on first run
spark.sql(f"CREATE SCHEMA IF NOT EXISTS `{CATALOG}`.`{SCHEMA}`")

spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {TARGET_TABLE} (
        id          STRING NOT NULL,
        cloud       STRING NOT NULL,
        title       STRING NOT NULL,
        description STRING,
        pub_date    DATE,
        url         STRING,
        status      STRING,
        feature     STRING,
        impact      STRING,
        categories  STRING,
        source      STRING,
        ingested_at TIMESTAMP
    ) USING DELTA
""")

# Pre-compute which rows are new vs. changed for the results display.
# This must happen BEFORE the merge because temp views are lazy.
DISPLAY_COLS = ["cloud", "title", "status", "feature", "impact", "pub_date", "source"]

target_hashed = (
    spark.table(TARGET_TABLE)
    .withColumn("t_hash", md5(concat_ws("|", *[coalesce(col(c), lit("")) for c in CONTENT_COLS])))
    .select(col("id").alias("t_id"), "t_hash")
)

staged = spark.table("release_notes_staged")
comparison = staged.alias("s").join(target_hashed, col("s.id") == col("t_id"), "left")

inserted_df = comparison.filter(col("t_hash").isNull()).select("s.*").drop("content_hash")
updated_df = (
    comparison.filter(col("t_hash").isNotNull() & (col("s.content_hash") != col("t_hash")))
    .select("s.*").drop("content_hash")
)

# Materialize to Pandas before the merge mutates the target
inserted_pdf = inserted_df.select(*DISPLAY_COLS).orderBy(col("pub_date").desc()).toPandas()
updated_pdf  = updated_df.select(*DISPLAY_COLS).orderBy(col("pub_date").desc()).toPandas()

n_inserted  = len(inserted_pdf)
n_updated   = len(updated_pdf)
rows_before = spark.table(TARGET_TABLE).count()
n_untouched = rows_before - n_updated

# MERGE: only update rows whose content actually changed
spark.sql(f"""
    MERGE INTO {TARGET_TABLE} AS tgt
    USING release_notes_staged AS src
    ON tgt.id = src.id
    WHEN MATCHED
        AND md5(concat_ws('|',
            coalesce(tgt.cloud,''),       coalesce(tgt.title,''),
            coalesce(tgt.description,''), coalesce(tgt.url,''),
            coalesce(tgt.status,''),      coalesce(tgt.feature,''),
            coalesce(tgt.impact,''),      coalesce(tgt.categories,''),
            coalesce(tgt.source,'')
        )) != src.content_hash
    THEN UPDATE SET
        tgt.cloud       = src.cloud,
        tgt.title       = src.title,
        tgt.description = src.description,
        tgt.pub_date    = src.pub_date,
        tgt.url         = src.url,
        tgt.status      = src.status,
        tgt.feature     = src.feature,
        tgt.impact      = src.impact,
        tgt.categories  = src.categories,
        tgt.source      = src.source,
        tgt.ingested_at = src.ingested_at
    WHEN NOT MATCHED THEN INSERT (
        id, cloud, title, description, pub_date,
        url, status, feature, impact, categories,
        source, ingested_at
    ) VALUES (
        src.id, src.cloud, src.title, src.description, src.pub_date,
        src.url, src.status, src.feature, src.impact, src.categories,
        src.source, src.ingested_at
    )
""")

rows_after = spark.table(TARGET_TABLE).count()

# Print summary
sep = "=" * 60
dash = "─" * 29
print(sep)
print("  MERGE RESULTS")
print(sep)
print(f"  Rows before:   {rows_before:>6,}")
print(f"  Rows after:    {rows_after:>6,}")
print(f"  {dash}")
print(f"  Inserted:      {n_inserted:>6,}")
print(f"  Updated:       {n_updated:>6,}")
print(f"  Untouched:     {n_untouched:>6,}")
print(sep)

## 6. Results

Display the rows that were updated or inserted in this run.

In [ ]:
print(f"🔄 Updated entries: {len(updated_pdf)}")
if len(updated_pdf) > 0:
    display(spark.createDataFrame(updated_pdf))
else:
    print("No entries were updated in this run.")

In [ ]:
print(f"🆕 Inserted entries: {len(inserted_pdf)}")
if len(inserted_pdf) > 0:
    display(spark.createDataFrame(inserted_pdf))
else:
    print("No new entries were inserted in this run.")